# Lesson 05：資料表合併與客群分析

學習目標：
- 建立完成訂單的事實表
- 使用 `customer_id` 合併顧客資料
- 檢查合併後列數與缺失值
- 依客群計算訂單數、顧客數、營收與平均訂單金額
- 建立「取得通路 x 客群」交叉分析表


## 流程大綱

1. 載入共用工具與原始資料
2. 建立完成訂單的 `facts`
3. 合併顧客表，補上 `segment` 與 `acquisition_channel`
4. 做合併品質檢查
5. 依客群彙總營收指標
6. 建立通路與客群的交叉表


## 1. 載入套件與資料


In [1]:
from common import ensure_packages, load_data, order_facts

ensure_packages()
data = load_data()

print(data.keys())


dict_keys(['customers', 'products', 'orders', 'order_items', 'sessions', 'events', 'ab_assignments'])


## 2. 建立訂單事實表

`order_facts(data)` 會先從 `order_items` 計算每筆訂單的 `line_revenue`，再合併 `orders`，最後只保留 `status == 'completed'` 的完成訂單。


In [2]:
facts = order_facts(data)

#print(f"facts rows: {len(facts):,}")
print(len(facts))
facts.head()


20413


,order_id,line_revenue,customer_id,order_date,status,payment_type
0,1,538.65,2323,2025-05-02,completed,wallet
1,2,4357.70,117,2024-07-14,completed,wallet
3,4,1193.40,1240,2024-07-08,completed,atm
4,5,13175.50,714,2024-08-12,completed,atm
5,6,3657.50,2005,2025-01-14,completed,card


## 3. 準備顧客表並檢查主鍵

合併前先確認 `customers['customer_id']` 是否唯一。若右表的 key 不唯一，合併後可能會讓列數膨脹，造成營收被重複計算。


In [3]:
customers = data["customers"]

print(f"customers rows: {len(customers):,}")
print(f"customers customer_id unique: {customers['customer_id'].is_unique}")
customers.head()


customers rows: 2,500
customers customer_id unique: True


,customer_id,signup_date,acquisition_channel,city,segment
0,1,2022-11-29,partner,Taipei,growth
1,2,2024-11-11,organic,Tainan,growth
2,3,2025-08-17,social,Hsinchu,vip
3,4,2025-06-09,organic,Taipei,growth
4,5,2024-07-23,referral,Taichung,new


## 4. 合併訂單與顧客資料

把完成訂單 `facts` 與顧客資料 `customers` 用 `customer_id` 做 left join。使用 `how='left'` 是為了保留所有完成訂單，即使某些訂單找不到顧客資料也不會被刪掉。


In [4]:
merged = facts.merge(customers, on="customer_id", how="left")

print(f"merged rows: {len(merged):,}")
print(f"segment null count: {merged['segment'].isna().sum():,}")
merged[["order_id", "customer_id", "line_revenue", "segment", "acquisition_channel"]].head()


merged rows: 20,413
segment null count: 0


,order_id,customer_id,line_revenue,segment,acquisition_channel
0,1,2323,538.65,growth,referral
1,2,117,4357.70,new,organic
2,4,1240,1193.40,new,ads
3,5,714,13175.50,growth,partner
4,6,2005,3657.50,new,ads


## 5. 合併品質檢查

重要的防呆檢查：合併後列數應該和合併前一樣。如果列數不同，通常代表 key 重複、join 條件錯誤，或資料表關係和預期不同。


In [6]:
assert len(merged) == len(facts), "警告:數量不一樣"
assert customers["customer_id"].is_unique, "customer_id 有重覆現象"

print("合併檢查通過：列數沒有改變，customer_id 在 customers 中也是唯一值。")


合併檢查通過：列數沒有改變，customer_id 在 customers 中也是唯一值。


## 6. 依客群計算筆數、平均營收與總營收

依照 `segment` 分組，計算 `line_revenue` 的筆數、平均值與總和。


In [7]:
seg = (
    merged.groupby("segment", as_index=False)["line_revenue"]
    .agg(["count", "mean", "sum"])
    .reset_index()
)
seg

,index,segment,count,mean,sum
0,0,growth,6155,5014.234297,30862612.10
1,1,new,12177,4949.503917,60270109.20
2,2,vip,2081,5112.483638,10639078.45


## 7. 加入更多商業指標

把客群分析擴充成更完整的表格：包含訂單數、獨立顧客數、總營收、平均訂單金額，以及每個客群的營收占比。


In [8]:
seg_analysis = (
    merged.groupby("segment", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        total_revenue=("line_revenue", "sum"),
        avg_order_value=("line_revenue", "mean"),
    )
    .round(2)
)

seg_analysis["revenue_share"] = (
    seg_analysis["total_revenue"] / seg_analysis["total_revenue"].sum() * 100
).round(1)

seg_analysis


,segment,order_count,unique_customers,total_revenue,avg_order_value,revenue_share
0,growth,6155,753,30862612.10,5014.23,30.3
1,new,12177,1490,60270109.20,4949.50,59.2
2,vip,2081,256,10639078.45,5112.48,10.5


## 8. 通路 x 客群交叉表

交叉表可以同時觀察兩個分類欄位。這裡用 `acquisition_channel` 當列、`segment` 當欄，分別計算訂單數與平均營收。


In [ ]:
cross = merged.pivot_table(
    values="line_revenue",
    index="acquisition_channel",
    columns="segment",
    aggfunc=["count", "mean"],
    margins=True,
).round(2)

cross

count                         mean                    \
segment             growth    new   vip    All   growth      new      vip   
acquisition_channel                                                         
ads                   1194   2640   403   4237  4871.73  4907.97  5046.19   
organic               1163   2532   391   4086  5134.43  4898.96  5420.78   
partner               1313   2389   465   4167  5070.05  5002.18  4743.32   
referral              1262   2520   425   4207  4981.61  4967.69  5173.21   
social                1223   2096   397   3716  5012.80  4980.98  5243.53   
All                   6155  12177  2081  20413  5014.23  4949.50  5112.48   

                              
segment                  All  
acquisition_channel           
ads                  4910.90  
organic              5015.92  
partner              4994.68  
referral             4992.62  
social               5019.50  
All                  4985.64

## 9. 解讀重點

看這類多表分析時，可以先問三個問題：

- 哪個客群貢獻最多總營收？
- 哪個客群的平均訂單金額較高？
- 不同取得通路帶來的客群組成是否不同？

如果總營收高但平均訂單金額普通，通常代表量大；如果平均訂單金額高但訂單數少，可能代表高價值但規模較小的客群。


## 10. 小練習

請建立一張表，依照 `acquisition_channel` 分組，計算：

- `order_count`：完成訂單數
- `unique_customers`：不重複顧客數
- `total_revenue`：總營收
- `avg_order_value`：平均訂單金額

提示：可以模仿前面的 `seg_analysis`，只要把分組欄位從 `segment` 改成 `acquisition_channel`。


In [10]:
# 請在這裡完成練習
channel_analysis = (
    merged.groupby("acquisition_channel", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        total_revenue=("line_revenue", "sum"),
        avg_order_value=("line_revenue", "mean"),
    )
    .round(2)
)

channel_analysis


,acquisition_channel,order_count,unique_customers,total_revenue,avg_order_value
0,ads,4237,511,20807499.60,4910.90
1,organic,4086,510,20495033.90,5015.92
2,partner,4167,511,20812828.40,4994.68
3,referral,4207,512,21003971.20,4992.62
4,social,3716,455,18652466.65,5019.50


## 11. 常見錯誤與延伸

常見錯誤：
- 合併後沒有檢查列數，導致 key 重複時營收被放大。
- 用 inner join 時不小心刪掉沒有對應顧客資料的訂單。

延伸練習：
- 在 `seg_analysis` 加上每位顧客平均營收：`revenue_per_customer = total_revenue / unique_customers`
- 將 `cross` 的 `aggfunc` 改成 `["count", "sum", "mean"]`，同時觀察訂單數、總營收與平均營收。
